# Phase 6 (post hoc): patient-relative beats, the full grid again (phase6_patient_template)

PREREGISTRATION §10 v6, P6. Dataset v3 = dataset v2 (v5) with every beat given minus the patient's local median beat
(±150 s, no labels). One notebook, one part per session:

| `PART` | accelerator | inputs | what it does |
|---|---|---|---|
| `data` | None (CPU) | `ecgaug-banks` | builds v3 and its banks, records the hashes; with `RUN_TESTS = True`, the full test suite first |
| `grid` | GPU | `ecgaug-banks` | the 6 cells without augmentation, the adoption rule, then (only if adopted) the other 90 cells |
| `diagnostic` | None | `ecgaug-banks`, `ecgaug-v1-checkpoints`, `ecgaug-v5-external`, v3 runs | train-versus-validation gap of the v1, v2, v3 networks; LDAs on fixed features (validation only) |
| `test` | None | `ecgaug-v1-checkpoints`, `ecgaug-v5-external`, v3 runs | one posthoc DS2 read (sets `ALLOW_TEST_POSTHOC=1`; logged; rebuilt from the raw records and hash-checked) |
| `external` | None | `ecgaug-v5-external`, v3 runs | INCART and SVDB with v3 beats |

Settings: **Internet on**. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`; `--part select` runs
wherever the ledger is, before `test`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "data"                         # "data", "grid", "diagnostic", "test" or "external"
DATABASE = "all"                      # external only: "incartdb", "svdb" or "all"
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_patient_template/patient_template.yaml"]
extra = ["--part", PART, "--device", DEVICE] + (["--database", DATABASE] if PART == "external" else [])
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_patient_template", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR, "--data-dir", INPUTS_DIR]
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
if PART == "test":
    env["ALLOW_TEST_POSTHOC"] = "1"   # the one posthoc DS2 read of PREREGISTRATION v6; it is logged in docs/TEST_ACCESS_LOG.md
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
